# Healthcare engagement

Define who counts as healthcare-engaged before `02_primary_incident_engagement`.

A contact day is a distinct calendar day with a recorded measurement or dispensing. The lookback is the 730 days before `KPRB_SAMPLE_DATE`, not including the sample date. A participant is engaged when that window contains at least one contact day. Participants with no contact day are the drop set.

Three definitions, from narrowest to broadest:

- `vitals_only`: BMI (`data/bmi.csv`) or blood pressure (`data/sbp.csv` and `data/dbp.csv`)
- `vitals_rx`: those vitals plus a medication dispensing (`data/rx.csv`)
- `all_contact`: vitals, dispensings, and any laboratory result (`data/labs.csv`)

`02_primary_incident_engagement` keeps participants with `all_contact` and `engaged == TRUE`. This notebook writes `data/healthcare_engagement/engage_list.rds`.


In [ ]:
suppressPackageStartupMessages({
  library(here)
  library(tidyverse)
  library(data.table)
})


# Cohort index date

The index date is `KPRB_SAMPLE_DATE` on `data/meta.csv`, the cohort `01_clin_preparation` starts from.


In [ ]:
# load meta data
meta <- read.csv(here("data/meta.csv"), row.names = 1) %>%
  mutate(
    StudyID = as.character(StudyID),
    KPRB_SAMPLE_DATE = as.Date(KPRB_SAMPLE_DATE)
  ) %>%
  select(StudyID, KPRB_SAMPLE_DATE)


# Contact sources

Each extract stores the contact date under a different column. Those columns are renamed to `CLIN_DATE` before counting.

| Source | File | Date column |
|---|---|---|
| BMI | `data/bmi.csv` | `MEASUREMENT_DATE` |
| Blood pressure | `data/sbp.csv`, `data/dbp.csv` | `MEASUREMENT_DATE` |
| Prescriptions | `data/rx.csv` | `RX_DATE` |
| Labs | `data/labs.csv` | `RESULT_DATE` |

Systolic and diastolic files replace the single blood-pressure extract. A day with either reading is one contact day.


In [ ]:
as_contact <- function(df, date_col) {
  stopifnot(all(c("StudyID", date_col) %in% names(df)))
  df %>%
    transmute(
      StudyID = as.character(StudyID),
      CLIN_DATE = .data[[date_col]]
    )
}

bmi <- read.csv(here("data/bmi.csv"), row.names = 1) %>%
  as_contact("MEASUREMENT_DATE")

bp <- bind_rows(
  read.csv(here("data/sbp.csv"), row.names = 1) %>% as_contact("MEASUREMENT_DATE"),
  read.csv(here("data/dbp.csv"), row.names = 1) %>% as_contact("MEASUREMENT_DATE")
)

rx <- read.csv(here("data/rx.csv")) %>%
  as_contact("RX_DATE")

labs <- read.csv(here("data/labs.csv")) %>%
  as_contact("RESULT_DATE")


# Definitions

Count distinct `CLIN_DATE` values in `[KPRB_SAMPLE_DATE - 730 days, KPRB_SAMPLE_DATE - 1 day]`. `engaged` is `n_contact_days >= 1`. People in `meta` with no row in the window stay in the table with `n_contact_days = 0`.


In [ ]:
LB_DAYS <- 730L
THRESHOLD <- 1L

build_engage <- function(foot_dfs, meta_dt, lb_days = LB_DAYS, threshold = THRESHOLD) {
  meta_dt <- data.table::as.data.table(meta_dt)
  meta_idx <- meta_dt[, .(StudyID, idx = data.table::as.IDate(KPRB_SAMPLE_DATE))]
  stopifnot(!anyNA(meta_idx$idx))

  foot <- data.table::rbindlist(lapply(names(foot_dfs), function(nm) {
    d <- data.table::as.data.table(foot_dfs[[nm]])
    stopifnot(all(c("StudyID", "CLIN_DATE") %in% names(d)))
    out <- d[, .(StudyID, CLIN_DATE = data.table::as.IDate(CLIN_DATE))]
    if (any(is.na(out$CLIN_DATE) & !is.na(d$CLIN_DATE)))
      stop(sprintf("Bad CLIN_DATE in %s", nm))
    out
  }), use.names = TRUE)

  foot <- meta_idx[foot, on = "StudyID", nomatch = 0L]
  foot <- foot[CLIN_DATE >= idx - lb_days & CLIN_DATE <= idx - 1L]

  eng <- foot[, .(n_contact_days = data.table::uniqueN(CLIN_DATE)), by = StudyID]
  engage <- eng[meta_dt[, .(StudyID)], on = "StudyID"]
  engage[is.na(n_contact_days), n_contact_days := 0L]
  engage[, engaged := n_contact_days >= threshold]
  engage[]
}

flags <- list(
  vitals_only = list(bmi = bmi, bp = bp),
  vitals_rx   = list(bmi = bmi, bp = bp, rx = rx),
  all_contact = list(bmi = bmi, bp = bp, rx = rx, labs = labs)
)

engage_list <- lapply(flags, build_engage, meta_dt = meta)

# 1. Does the drop-set size move across definitions?
sapply(engage_list, function(e) sum(e$n_contact_days == 0L))

# 2. Check if the same people get dropped across definitions
zero_sets <- lapply(engage_list, function(e) e[n_contact_days == 0L, StudyID])
zero_overlap <- outer(seq_along(zero_sets), seq_along(zero_sets), Vectorize(function(i, j)
  length(intersect(zero_sets[[i]], zero_sets[[j]]))))
dimnames(zero_overlap) <- list(names(zero_sets), names(zero_sets))
zero_overlap


# Export

`engage_list` is a named list of three tables (`StudyID`, `n_contact_days`, `engaged`). `02_primary_incident_engagement` reads `all_contact` from this file.


In [ ]:
dir.create(here("data/healthcare_engagement"), showWarnings = FALSE, recursive = TRUE)
saveRDS(engage_list, file = here("data/healthcare_engagement/engage_list.rds"))
